In [16]:
%load_ext autoreload
%autoreload 2

In [17]:
from pathlib import Path
import os
import socket
import sys
from dotenv import load_dotenv
from pathlib import Path
import sys
import wandb


In [49]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

In [18]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
env_path = (project_root / "env.env").resolve()
status = load_dotenv(dotenv_path=env_path, override=True)

print("env file:", env_path)
print("load_dotenv returned:", status)
data_dir = project_root / "datasets"

Project Root: /coc/scratch/vchopra/learning_env/Deep-Learning
env file: /coc/scratch/vchopra/learning_env/Deep-Learning/env.env
load_dotenv returned: True


In [19]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "inara" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [ ]:

from datasets import Dataset as hf_data
from torch.utils.data import Dataset
import tiktoken
import numpy, tqdm
import matplotlib.pyplot as plt
%matplotlib inline

In [21]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [22]:

api_key = os.environ.get("WANDB_API_KEY")
assert api_key, "WANDB_API_KEY was not loaded from env.env"

wandb.finish()
wandb.login(key=api_key, relogin=True)

api = wandb.Api()
print("Logged-in W&B username:", api.viewer.username)
print("Default W&B entity:", api.default_entity)
print("WANDB_API_KEY loaded from environment:", bool(os.getenv("WANDB_API_KEY")))


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /nethome/dgarg39/.netrc
wandb: Currently logged in as: vedaangchopra (vedaangchopra_gatech) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Logged-in W&B username: vedaangchopra
Default W&B entity: vedaangchopra_gatech
WANDB_API_KEY loaded from environment: True


In [24]:
from model_training.framework.data.post_train.instruction_tuning.raw_data.simple_example.instruct_data import download_and_load_file
from model_training.framework.data.post_train.instruction_tuning.prompt_style.styles import SFTDatasetPrepare, AlpacaFormatOld
from model_training.framework.data.post_train.dataloader import InstructionDataset, custom_collate_fn, create_instruction_dataloader

## Fix this import

# raise Exception("Not Running on Pace")
    # from src.model import GPT

In [25]:
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, f"{props.total_memory / 1024**3:.1f} GiB")

GPU count: 8
0 NVIDIA A40 44.4 GiB
1 NVIDIA A40 44.4 GiB
2 NVIDIA A40 44.4 GiB
3 NVIDIA A40 44.4 GiB
4 NVIDIA A40 44.4 GiB
5 NVIDIA A40 44.4 GiB
6 NVIDIA A40 44.4 GiB
7 NVIDIA A40 44.4 GiB


### Step-1: - Dataset Load and Basic Inference

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [37]:
def return_model_tokenizer(MODEL_NAME, device):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    model = model.to(device).eval()
    print("Device:", device)
    return model, tokenizer

model, tokenizer = return_model_tokenizer(MODEL_NAME, device)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Device: cuda


#### Sample Example: - Just for reference

In [42]:
def prepare_input_tokens(messages, tokenizer, device):
    if messages==None:
        messages = [
        {
            "role": "user",
            "content": (
                "Table:\n"
                "Service A | Active\n"
                "Service B | Inactive\n"
                "Question: What is the status of Service B? "
                "Answer with one word."
            )
        }
    ]
        
    input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
    ).to(device)
    
    return input_ids

input_ids = prepare_input_tokens(None, tokenizer, device)
input_ids

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,   2556,    510,   1860,
            362,    760,   9905,    198,   1860,    425,    760,    758,   3028,
            198,  14582,     25,   3555,    374,    279,   2639,    315,   5362,
            425,     30,  21806,    448,    825,   3409,     13, 151645,    198,
         151644,  77091,    198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [ ]:
with torch.no_grad():
    output_ids = model.generate(
        input_ids['input_ids'],
        max_new_tokens=32,
        do_sample=True,
        temperature=1.0,
        top_k=0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id
    )
output_ids

tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,   2556,    510,   1860,
            362,    760,   9905,    198,   1860,    425,    760,    758,   3028,
            198,  14582,     25,   3555,    374,    279,   2639,    315,   5362,
            425,     30,  21806,    448,    825,   3409,     13, 151645,    198,
         151644,  77091,    198,  72214, 151645]], device='cuda:0')

In [55]:
prompt_len = input_ids['input_ids'].shape[1]
generated_ids = output_ids[:, prompt_len:]
print(prompt_len, len(output_ids[0]), )

57 59


In [56]:
print(tokenizer.decode(
    generated_ids[0], skip_special_tokens=True
))

Inactive


### Step-2: - Understand the token probability of each token

In [61]:
with torch.no_grad():
    logits = model(output_ids).logits

# print(logits.shape, prompt_len-1)
action_logits = logits[:, prompt_len-1:-1, :]
log_probs = F.log_softmax(action_logits, dim=-1)

In [68]:
# generated_ids.shape, generated_ids.squeeze(0).shape, generated_ids.unsqueeze(-1).shape

In [71]:
token_log_probs = log_probs.gather(
    dim=-1, 
    index=generated_ids.unsqueeze(-1)
)
assert token_log_probs.shape == generated_ids.shape

token_log_probs

AssertionError: 

In [72]:
for token_id, logp in zip(
    generated_ids[0], token_log_probs[0]
):
    token = tokenizer.decode([token_id.item()])
    print(repr(token), "log_prob:", round(logp.item(), 4))


'Inactive' log_prob: -0.0
'<|im_end|>' log_prob: -0.0001
